# Day 8 — Model Evaluation & Residual Diagnostics
## M5 Walmart Demand Forecasting (LightGBM)

**Series context:** This notebook is Day 8 of a day-by-day forecasting project built on the M5 Walmart sales dataset. Days 1–7 covered data cleaning, feature engineering (lags, rolling statistics, price dynamics, calendar/SNAP effects), and progressive LightGBM model tuning. The trained model and its supporting artifacts (feature list, categorical columns, validation split, metrics) were persisted with `joblib` at the end of Day 7.

**Goal of this notebook:** Load the finalized Day 7 model, generate predictions on the held-out (time-based) validation set, and rigorously diagnose forecast quality — not just "is the RMSE low," but *where* the model succeeds, *where* it fails, and what that failure pattern means operationally for a retailer managing inventory across thousands of item–store combinations.

**Why this matters commercially:** A demand forecast is only useful insofar as it drives good inventory decisions. Two models can have identical RMSE yet imply very different business risk — one that is well-calibrated but noisy is safer to act on than one that is biased in a single direction. This notebook focuses on separating *accuracy* (how large are the errors) from *bias* (which direction do the errors lean) and *tail risk* (how bad are the worst misses), since these three dimensions map directly to distinct inventory failure modes: safety-stock sizing, systematic overstock/understock, and stockout exposure during demand spikes.

**What is *not* changed here:** All model training logic, the trained model artifact, and every cell's original output are preserved exactly as produced in the source run. This pass only adds explanatory documentation and interpretation around the existing analysis.


## 1. Load Model Artifacts from Day 7

The cells below reload everything the Day 7 training pipeline persisted to disk:

- **Trained LightGBM model** — the final selected model after tuning.
- **Feature list** — the exact ordered set of columns the model expects at inference time.
- **Categorical columns** — features LightGBM treats natively as categorical (native categorical handling avoids one-hot blow-up across high-cardinality fields like `item_id` and `store_id`).
- **Best metrics** — the RMSE/MAE recorded at the end of training, used here as a reference baseline to confirm the reloaded model reproduces the same validation performance.

**Why reload from disk instead of reusing in-memory objects?** In a production-style workflow, model *training* and model *evaluation/serving* are treated as separate stages, often run in different sessions, environments, or by different team members. Reloading from serialized artifacts (`.pkl` / `.json`) mirrors how a model would actually be handed off from a training pipeline to a deployment or monitoring pipeline, and confirms the model is fully self-contained and reproducible outside the training notebook.


In [1]:
import joblib
import json

**Loading the trained model.** This is the LightGBM regressor selected as "best" during Day 7's tuning process — i.e., the configuration that minimized validation RMSE across the hyperparameter search.

In [2]:
model = joblib.load(r"../models/tuned/best_lightgbm_model.pkl")


**Loading the feature schema.** The model must see features in a consistent structure at inference time. This list includes:

- *Identifiers*: `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id`
- *Calendar*: `weekday`, `wday`, `month`, `year`, `event_name_1/2`, `event_type_1/2`
- *Promotional*: `snap_CA`, `snap_TX`, `snap_WI` (SNAP benefit-disbursement days, a known demand driver in the M5 dataset)
- *Price dynamics*: `sell_price`, `price_lag_1`, `price_diff_1`, `price_pct_change`, `price_max`, `price_relative_to_max`
- *Demand history*: `lag_1/7/14/28` and `rolling_mean/std_7/14/28`

**Technical insight:** The heavy reliance on lag and rolling-window features means this model is fundamentally an *autoregressive* demand model — it learns "what tends to happen next given recent sales behavior," conditioned on price and calendar context. This has a direct consequence for evaluation: performance will typically be strongest on stable, regularly-selling items and weakest on intermittent or newly-launched items where recent history is sparse or unrepresentative — a pattern we investigate later in this notebook.

In [3]:
feature_names = joblib.load(r"../models/tuned/feature_names.pkl")
feature_names


['item_id',
 'dept_id',
 'cat_id',
 'store_id',
 'state_id',
 'wm_yr_wk',
 'weekday',
 'wday',
 'month',
 'year',
 'event_name_1',
 'event_type_1',
 'event_name_2',
 'event_type_2',
 'snap_CA',
 'snap_TX',
 'snap_WI',
 'sell_price',
 'lag_1',
 'lag_7',
 'lag_14',
 'lag_28',
 'rolling_mean_7',
 'rolling_std_7',
 'rolling_mean_14',
 'rolling_std_14',
 'rolling_mean_28',
 'rolling_std_28',
 'is_weekend',
 'price_lag_1',
 'price_diff_1',
 'price_pct_change',
 'price_max',
 'price_relative_to_max']

**Loading categorical column definitions.** Explicitly marking categorical features lets LightGBM use its native histogram-based categorical splitting rather than relying on arbitrary numeric encoding, which typically improves both accuracy and training speed for high-cardinality identifiers like `item_id` and `store_id`.

In [4]:
categorical_cols = joblib.load(r"../models/tuned/categorical_cols.pkl")

## 2. Reference Metrics from Training

Before regenerating predictions, we load the RMSE/MAE that were recorded during Day 7 training. These serve as a **reproducibility check** — if predictions generated in this notebook don't reconcile with these numbers, that would flag a data or environment mismatch (e.g. wrong validation split, feature order mismatch, or a different model version) before any downstream analysis is trusted.

In [5]:
import json

with open(
    r"../models/tuned/best_metrics.json",
    "r"
) as f:
    metrics = json.load(f)

print(metrics)

{'RMSE': 1.2596662880136444, 'MAE': 0.4872918586763616}


In [6]:
print("Model Loaded Successfully")
print("Metrics:", metrics)
print("Features:", len(feature_names))
print("Categorical Columns:", len(categorical_cols))

Model Loaded Successfully
Metrics: {'RMSE': 1.2596662880136444, 'MAE': 0.4872918586763616}
Features: 34
Categorical Columns: 9


### RMSE & MAE — What These Numbers Actually Mean

The reloaded model reports **RMSE ≈ 1.26** and **MAE ≈ 0.49** units sold, per item–store–day, on the validation set.

**MAE (Mean Absolute Error) — the "typical" error.**
MAE treats every error equally regardless of size, so it answers a simple operational question: *"On an average day, for an average item at an average store, how many units off is the forecast?"* An MAE of 0.49 means the forecast is, on average, well under one unit away from actual demand — which sounds excellent, but must be read in context: M5 daily unit sales at the item–store level are dominated by **low-volume, intermittent demand** (many days with 0–2 units sold). A sub-1-unit MAE is expected and even necessary at this granularity; it does not by itself imply the model is "production ready" for higher-volume or spike scenarios.

**RMSE (Root Mean Squared Error) — the "worst-case-sensitive" error.**
RMSE squares each error before averaging, which disproportionately penalizes large misses. Here, **RMSE (1.26) is roughly 2.6× MAE (0.49)**. This gap is itself a diagnostic signal: if all errors were similar in size, RMSE and MAE would be close together. A large RMSE/MAE ratio indicates the error distribution is **heavy-tailed** — most days are forecast very accurately, but a minority of days carry disproportionately large errors. This is the statistical fingerprint of demand spikes (promotions, SNAP days, holidays) and intermittent-demand items, and it's confirmed later in the residual distribution (max error of +615 units on an otherwise sub-1-unit-error series).

**Business takeaway:** MAE tells inventory planners what to expect on a "normal" day (safe to plan tight buffers). RMSE tells risk/finance teams what to expect in the worst case (informs safety-stock and contingency planning). Reporting *only* RMSE or *only* MAE — as many forecasting projects do — hides this bimodal risk profile; both are needed together to correctly size inventory buffers.

## 3. Generate Predictions on the Held-Out Validation Set

We reload the time-based validation split (`X_valid`, `y_valid`) that Day 7 carved out and score the model on it.

**Why a time-based split matters here:** Unlike a random train/test split, a time-based split ensures the validation period comes *strictly after* the training period in calendar time. This is essential for demand forecasting because it prevents "look-ahead" leakage — the model is never validated on information (e.g. lagged/rolling features) that implicitly encodes future demand it wouldn't have access to in a real deployment. A model evaluated this way gives a much more honest estimate of production performance than a randomly-shuffled split would.

**Business value:** This is the closest proxy available, before live deployment, to how the model would actually perform if used today to forecast tomorrow's demand — making the metrics computed from this split directly actionable for planning purchase orders, replenishment schedules, and staffing.

In [7]:
import joblib

# Load model
model = joblib.load(
    r"../models/tuned/best_lightgbm_model.pkl"
)

# Load validation data
X_valid = joblib.load(
    r"../models/tuned/X_valid.pkl"
)

y_valid = joblib.load(
    r"../models/tuned/y_valid.pkl"
)

# Predictions
y_pred = model.predict(X_valid)

# Save predictions
joblib.dump(
    y_pred,
    r"../models/tuned/y_pred.pkl"
)

print("Prediction Shape:", y_pred.shape)
print("Validation Shape:", y_valid.shape)

Prediction Shape: (4524000,)
Validation Shape: (4524000,)


## 4. Diagnostic Deep-Dive: Actual vs. Predicted

The next several cells build up a comparison table (`Actual`, `Predicted`, `Error = Actual − Predicted`) and progressively filter it — first an unfiltered head, then filtering to non-zero, then strictly positive actual demand.

**Why this progressive filtering approach?** M5 sales data is heavily **zero-inflated** — a large share of item–store–day combinations have zero sales on any given day (out of stock, low-velocity SKU, not yet launched at that store, etc.). If we only look at an unfiltered sample, it will be dominated by trivial zero-vs-near-zero predictions that tell us little about real forecasting risk. Filtering to `Actual > 0` isolates the rows that actually matter operationally — the days where a customer wanted to buy something — which is exactly where forecast errors translate into real stockout or overstock consequences.

In [8]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Actual": y_valid.values[:20],
    "Predicted": y_pred[:20]
})

comparison_df["Error"] = (
    comparison_df["Actual"]
    - comparison_df["Predicted"]
)

comparison_df

,Actual,Predicted,Error
0,0,0.028702,-0.028702
1,0,0.028702,-0.028702
2,0,0.008814,-0.008814
3,0,0.010080,-0.010080
4,0,0.007301,-0.007301
5,0,0.013225,-0.013225
6,0,0.022243,-0.022243
7,0,0.036207,-0.036207
8,0,0.036207,-0.036207
9,0,0.021257,-0.021257


In [9]:
comparison_df.head(24)

,Actual,Predicted,Error
0,0,0.028702,-0.028702
1,0,0.028702,-0.028702
2,0,0.008814,-0.008814
3,0,0.010080,-0.010080
4,0,0.007301,-0.007301
5,0,0.013225,-0.013225
6,0,0.022243,-0.022243
7,0,0.036207,-0.036207
8,0,0.036207,-0.036207
9,0,0.021257,-0.021257


In [10]:
comparison_df[comparison_df["Actual"] != 0].head(20)

,Actual,Predicted,Error


In [11]:
comparison_df[comparison_df["Actual"] > 0].head(20)

,Actual,Predicted,Error


In [12]:
comparison_df = pd.DataFrame({
    "Actual": y_valid.values,
    "Predicted": y_pred
})

comparison_df["Error"] = (
    comparison_df["Actual"]
    - comparison_df["Predicted"]
)

comparison_df[comparison_df["Actual"] > 0].head(20)

,Actual,Predicted,Error
857,1,0.007301,0.992699
858,1,0.305265,0.694735
860,1,0.290271,0.709729
864,3,0.250181,2.749819
868,4,0.505362,3.494638
869,2,1.034758,0.965242
872,2,0.509247,1.490753
873,1,0.674721,0.325279
875,1,0.897815,0.102185
876,1,1.005977,-0.005977


### Reading the Non-Zero Demand Sample

Once we filter across the *full* validation set for `Actual > 0`, real demand-forecasting behavior becomes visible. A few illustrative rows:

- Rows with `Actual = 1` and `Predicted ≈ 0.7–1.0` show the model tracking low, steady demand closely — small errors, low operational impact.
- Row with `Actual = 4`, `Predicted ≈ 0.51` (`Error ≈ +3.49`) is a clear **under-forecast**: the model substantially underestimated a demand spike. In practice, this is the failure mode that causes **stockouts** — inventory ordered against the forecast would fall short of what customers actually wanted.
- Row with `Actual = 1`, `Predicted ≈ 1.18` (`Error ≈ −0.18`) is a mild **over-forecast** — inventory ordered slightly exceeds demand, a low-cost error compared to under-forecasting.

**Forecasting interpretation:** Errors are not symmetric in business consequence even when they're symmetric in magnitude. An under-forecast of 3–4 units on a fast-moving item can mean a stocked-out shelf and a lost sale (plus potential customer churn to a competitor), while an over-forecast of similar magnitude typically just means a few extra units sitting in inventory — costly, but rarely catastrophic. This asymmetry is why raw RMSE/MAE, while useful for model selection, are insufficient on their own for inventory decision-making — bias direction (next section) and tail behavior matter just as much.

## 5. Bias Analysis — Is the Model Systematically Over- or Under-Forecasting?

RMSE and MAE describe error *magnitude*, but not error *direction*. Two models can share the same RMSE while one is perfectly centered around zero error and the other is consistently skewed high or low — with very different inventory consequences. **Bias** captures this directional signal:

$$\text{Bias} = \frac{1}{n}\sum (\text{Actual}_i - \text{Predicted}_i)$$

- **Bias ≈ 0** → the model is well-calibrated on average; errors cancel out across the portfolio.
- **Bias < 0** (Actual < Predicted on average) → the model **over-forecasts** systematically → excess inventory, higher holding costs, markdowns/waste risk (especially for perishable or seasonal goods).
- **Bias > 0** (Actual > Predicted on average) → the model **under-forecasts** systematically → insufficient inventory → recurring stockouts, lost sales, potential SLA/service-level breaches with retail partners.

In [13]:
bias = (y_valid - y_pred).mean()

print("Bias:", bias)

Bias: -0.012075719936628242


### Interpreting the Result

The computed bias is **≈ −0.0121 units**, i.e. essentially zero relative to the demand scale in this dataset (recall MAE ≈ 0.49). The slight negative sign indicates a very mild **over-forecasting tendency** on average across the ~4.5M validation observations — the model predicts marginally more than actual demand more often than the reverse.

**Business read:** A bias this close to zero, at this scale, is a strong result — it means the model is not systematically padding or shorting the portfolio, so aggregate purchase orders built from these forecasts should not drift over or under actual demand over time. However, "near-zero on average" can still mask **localized bias** at the segment level (e.g. a specific category, store, or region consistently over/under-forecast even though the portfolio-wide average cancels out). A natural next step for this project would be to compute bias grouped by `store_id`, `dept_id`, or `state_id` to check for such pockets before trusting the model for store-level or category-level replenishment decisions rather than only portfolio-wide ones.

## 6. Residual Analysis — Understanding the Full Error Distribution

Bias tells us about the *average* error; residual analysis tells us about the *shape* of the error distribution — its spread, skew, and extremes. This matters because inventory safety-stock is typically sized off the **spread** of forecast error (e.g. error standard deviation), not just its average, and extreme residuals reveal exactly how bad the worst-case forecasting misses can get.

In [14]:
residuals = y_valid - y_pred

print(residuals.describe())

count    4.524000e+06
mean    -1.207572e-02
std      1.259609e+00
min     -9.151582e+01
25%     -3.369233e-01
50%     -3.961410e-02
75%     -1.983829e-04
max      6.153893e+02
Name: sales, dtype: float64


### Interpreting the Residual Distribution

| Statistic | Value | Interpretation |
|---|---|---|
| Mean | −0.012 | Matches the bias computed above — negligible systematic skew |
| Std Dev | 1.260 | Typical spread of forecast error around the mean — directly informs safety-stock sizing |
| Min | −91.52 | Worst single **over-forecast**: predicted ~92 units more than were actually sold |
| 25th pct | −0.337 | A quarter of forecasts over-predict by more than ~0.34 units |
| Median | −0.040 | Center of the distribution is very close to zero, slightly over-forecasting |
| 75th pct | ~0.000 | Three-quarters of forecasts under-predict by less than ~0.0002 units (i.e. the bulk of the distribution is over-forecast or exactly on target) |
| Max | +615.39 | Worst single **under-forecast**: actual demand exceeded prediction by over 615 units in one instance |

**Forecasting-specific insight — the tails tell the real story.** The middle 50% of residuals (25th–75th percentile) span only about 0.34 units — the model is extremely precise for the bulk of typical, low-volume item–store–days. But the extremes are enormous by comparison: a −91.5 to +615.4 unit range. This is the classic signature of **demand spikes** — promotional events, SNAP-driven surges, or holiday effects — that occur rarely but drive very large absolute errors when they do. These rare events are exactly what inflates RMSE far above MAE, as noted in Section 2.

**Under-forecasting vs. over-forecasting risk, in this distribution:**
- The **maximum positive residual (+615)** represents a severe **under-forecast** on what was almost certainly a large promotional or event-driven spike (e.g. a major SNAP or holiday demand surge for a high-velocity item). If inventory were ordered strictly against this forecast, the result would be a significant **stockout** — empty shelves during exactly the high-traffic period the retailer most needs to capture demand.
- The **minimum negative residual (−91.5)** represents a severe **over-forecast** — inventory would be ordered for demand that never materialized, resulting in **overstock**: tied-up capital, markdown risk, and (for perishable categories) potential write-offs.
- Because the distribution's tails are asymmetric in *magnitude* (max +615 vs. min −92) while the median sits slightly negative, the practical risk profile is: **frequent, small over-forecasts** (low-cost, manageable via routine markdown/clearance) versus **rare, extreme under-forecasts** (high-cost, since stockouts during genuine demand spikes directly cost sales and can damage retailer trust).

**Recommended inventory policy implication:** Given this asymmetry, a defensible planning strategy is to accept the (cheap) tendency toward mild over-forecasting on typical days, while building explicit **spike-detection or event-flagging logic** (e.g. flagging SNAP days, promotional calendars, or holidays) to add a deliberate safety margin specifically around those windows — rather than uniformly inflating safety stock everywhere, which would erode the model's otherwise tight, well-calibrated performance on ordinary days.

## 7. Summary & Next Steps

**Key findings from this evaluation:**
- **Accuracy:** MAE ≈ 0.49 units and RMSE ≈ 1.26 units on ~4.5M validation observations — strong performance for item–store–day level demand, consistent with the intermittent, low-volume nature of the majority of SKU–store combinations in this dataset.
- **Calibration:** Portfolio-wide bias ≈ −0.012 units — negligible systematic skew; the model is not structurally over- or under-forecasting at the aggregate level.
- **Tail risk:** The RMSE/MAE gap and residual range (−91.5 to +615.4) confirm a heavy-tailed error distribution driven by rare demand spikes, not by the model being uniformly imprecise.
- **Business risk asymmetry:** Errors skew toward frequent, low-cost over-forecasting on typical days, with rare but high-cost under-forecasting during demand spikes — the latter being the primary stockout risk to manage operationally.

**Recommended next steps (Day 9+):**
1. Break bias and RMSE down by `store_id`, `dept_id`/`cat_id`, and `state_id` to check for localized miscalibration hidden by the portfolio-wide averages.
2. Isolate and profile the largest-residual observations (e.g. top 0.1% by `|Error|`) to confirm they correspond to identifiable events (SNAP days, calendar events, promotions) rather than data-quality issues.
3. Evaluate a service-level-oriented metric (e.g. weighted quantile loss / pinball loss at a high quantile) alongside RMSE/MAE, since retail replenishment usually cares more about avoiding stockouts than about minimizing average error symmetrically.
4. Consider a lightweight spike-classification flag (event/SNAP calendar features already exist in the schema) feeding into a dynamic safety-stock multiplier, rather than a single global buffer.


In [16]:
import numpy as np
error_df = pd.DataFrame({
    "actual": y_valid,
    "predicted": y_pred,
    "error": y_pred - y_valid,
    "abs_error": np.abs(y_pred - y_valid)
})

error_df.to_pickle("../data/processed/analysis/error_df.pkl")

All Day 7 artifacts loaded successfully.
